# CS604 Colab launcher — Ana: LoRA rank r16 vs r24 (all 10,000 classes, 60 min each)

Full-scale check of the pilot rank result (r24 +0.66 over r16 on the 1,000-class pilot, repeated at −0.54). Spec: `configs/lora_r16-vs-r24_336px_all-10000-classes_60min.yaml`.

Same holdout as the all-class LoRA run and the controlled PEFT comparison (all 10,000 classes, 5 per class held out, subset seed 0) and the same recipe (`configs/peft_336px_shared-recipe_proto-cosine_no-extras.yaml`: 336 px, prototype cosine head from the fit images, head lr 3e-3, no extras, plain crop 0.9), but **60 training minutes per arm** (5% warmup + cosine fitted to 60 min). Only the rank changes; alpha = rank.

| Arm | LoRA | Backbone % |
|---|---|---|
| A_lora_r24 | r24, qkv/proj/fc1/fc2, all 12 blocks | 4.12 |
| B_lora_r16 | r16, same targets | 2.74 |
| A_lora_r24_seed1, B_lora_r16_seed1 | optional second seed | |

About 65–70 min per arm on an **A100** (do not mix GPU types: arms get equal minutes), so the two main arms take ~2.3 h plus the data download. If a session drops, Run all again: finished arms are skipped and an interrupted arm resumes.

Runtime: **A100 GPU**. Use **Runtime → Run all**.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
METHOD = "lora_r24_336px_qkv-proj-fc_extras_proto-cosine_2.4h"  # base config; only used to build prototypes_336.pt if it is missing
PROTO_PT = f"{DRIVE_OUT}/prototypes_336.pt"
SPEC = "configs/lora_r16-vs-r24_336px_all-10000-classes_60min.yaml"
# Main comparison. Add "A_lora_r24_seed1", "B_lora_r16_seed1" for a second seed (another ~2.3 h).
ONLY = ["A_lora_r24", "B_lora_r16"]  # empty = every arm in the spec, seeds included
PILOT_DATA_DIR = "/content/pilot_data"
FEATURE_DIR = "/content/features_pilot"  # local disk
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()


## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 4b. Build class prototypes at 336 (once)

Same as Van's prototype head, but built from 336 features (built with `configs/{METHOD}.yaml`, so at its `img_size`). Run **once**; about 20–25 min on an A100. Takes a while (embeds train images). Skip if the file already exists on Drive.

In [ ]:
from pathlib import Path

if Path(PROTO_PT).is_file():
    print("Already have", PROTO_PT)
else:
    !python scripts/build_prototypes.py --config configs/{METHOD}.yaml --data_root {DATA_DIR} --output {PROTO_PT}

## 4c. Reuse the fit-image prototypes (skips ~20 min)

The all-class holdout split (seed 0) is the same as in the earlier all-class runs, so their fit-image prototypes are copied in if one exists. Otherwise `scripts/ablation.py` rebuilds them.


In [ ]:
import shutil
from pathlib import Path

target = Path(DRIVE_OUT) / f"ablation_{Path(SPEC).stem}" / "prototypes_pilot_fit.pt"
sources = [Path(DRIVE_OUT) / f"ablation_{name}" / "prototypes_pilot_fit.pt"
           for name in ("ablation_peft_controlled", "lora_r24_336px_no-extras_all-10000-classes_40fit-5holdout_144min")]
if target.is_file():
    print("Already have", target)
else:
    found = next((s for s in sources if s.is_file()), None)
    if found:
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(found, target)
        print("Copied", found, "->", target)
    else:
        print("No earlier prototypes_pilot_fit.pt found; the ablation will build it")


## 5. Ablation

In [ ]:
import subprocess
import sys

proc = subprocess.run([sys.executable, "-u", "scripts/ablation.py", "--spec", SPEC,
                       "--data_root", DATA_DIR, "--output_dir", DRIVE_OUT,
                       "--pilot_data_dir", PILOT_DATA_DIR, "--feature_dir", FEATURE_DIR]
                      + (["--only", *ONLY] if ONLY else []))
if proc.returncode != 0:
    end_runtime(f"ablation failed (exit {proc.returncode}); see the output above")
    raise SystemExit("ablation failed")

## 6. Results

`delta_vs_A_points`: change in pilot holdout top-1 (percentage points) against the full recipe. Negative = turning that piece off hurts, so that piece helped. Treat differences under ~0.3 points as ties (one seed).

In [ ]:
import json
from pathlib import Path

import pandas as pd

# Read the per-arm markers, not ablation.csv: with ONLY, each session rewrites ablation.csv
# with its own arms, while arms/*.json holds every finished arm.
arms_dir = Path(DRIVE_OUT) / f"ablation_{Path(SPEC).stem}" / "arms"
table = pd.DataFrame([json.loads(p.read_text()) for p in sorted(arms_dir.glob("*.json"))])
table["seed"] = table["arm"].str.extract(r"seed(\d+)$")[0].fillna("42")
table["rank"] = table["arm"].str.extract(r"_r(\d+)")[0].astype(int)
ref = table[table["rank"] == 24].set_index("seed")["final_val_acc"]
table["delta_vs_r24_points"] = (100 * (table["final_val_acc"] - table["seed"].map(ref))).round(2)
display(table[["arm", "rank", "seed", "final_val_acc", "delta_vs_r24_points", "final_val_loss",
               "backbone_pct", "peak_vram_gb", "training_min", "wall_min"]])


## 7. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")